In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 00. 이 책의 학습 방법과 환경 설정

> **학습 목표**
> - LLM 시대에 수학이 왜 중요한지 설명할 수 있다
> - 이 책의 상향식 구성과 학습법을 이해한다
> - Colab CPU/GPU 런타임을 자유롭게 전환할 수 있다
> - 공통 유틸리티 패키지 `llm_math`를 사용할 수 있다

## 0.1 왜 "수학부터" LLM인가?

ChatGPT, GPT-4, LLaMA, Claude... LLM(대형 언어 모델)은 2022년 이후 세상을 바꿨다.
하지만 대부분의 개발자는 단순히 API를 호출할 뿐, 모델 내부에서 무슨 일이 일어나는지 모른다.

이 책은 **한 줄 수식에서 시작해 GPT를 직접 만드는 것**을 목표로 한다.
- 행렬곱 $AB$가 왜 어텐션의 핵심인가?
- 소프트맥스 $\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$가 확률 분포를 어떻게 만드는가?
- 역전파는 체인룰 $\frac{d}{dx}f(g(x)) = f'(g(x)) g'(x)$를 어떻게 기계적으로 적용하는가?

이 질문들에 답하면, LLM을 "사용"하는 것을 넘어 "이해"하고 "만들" 수 있다.

## 0.2 이 책의 구성 원리

```
수학 정의 → 직관(그림) → 수식 유도 → 코드 구현 → 벤치마크 → 응용
```

모든 장은 이 6단계로 구성된다.

## 0.3 환경 설정

Colab에서 이 노트북을 실행한다고 가정한다. 아래 셀을 실행하여 환경을 점검하자.


In [ ]:
# 환경 점검
import sys
print(f"Python: {sys.version.split()[0]}")

# 핵심 라이브러리
import numpy as np
print(f"NumPy: {np.__version__}")

import matplotlib
print(f"Matplotlib: {matplotlib.__version__}")

import scipy
print(f"SciPy: {scipy.__version__}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch: NOT installed (Part 2 이전에는 필요 없음)")


### llm_math 패키지 로드

이 책의 모든 노트북은 공통 유틸리티 패키지 `llm_math`를 사용한다.
첫 코드 셀에서 자동으로 불러온다. (직접 확인해보자.)


In [ ]:
# preamble에서 로드한 llm_math 상태 확인
if _LLM_MATH_OK:
    print("✓ llm_math 패키지 로드 성공")
    print(f"  - viz: Visualization 헬퍼 ({viz.__name__})")
    print(f"  - bench: CPU vs GPU 벤치마크 헬퍼 ({bench.__name__})")
    print(f"  - data: 미니 Data셋 로더 ({data.__name__})")
else:
    print("⚠ 로드 실패. 아래 명령으로 레포를 클론하세요:")
    print("  !git clone https://bit.ly/4f1YusE")
    print("  %cd llm-math-book")
    print("  !bash colab_setup.sh")


## 0.4 첫 벤치마크 — 작은 행렬곱으로 CPU/GPU 비교 맛보기

이 책의 핵심 테마 중 하나는 **CPU와 GPU의 속도 차이를 직접 체감**하는 것이다.
간단한 행렬곱으로 그 맛을 먼저 보자.


In [ ]:
# 첫 벤치마크: 행렬곱 CPU vs GPU
import time
import numpy as np

def bench_np_matmul(n, repeat=5):
    """NumPy Matrix곱 Time Measurement (CPU)."""
    A = np.random.randn(n, n).astype(np.float32)
    B = np.random.randn(n, n).astype(np.float32)
    # warmup
    for _ in range(2):
        _ = A @ B
    # Measurement
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        _ = A @ B
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)
    return np.mean(times), np.std(times)

# Matrix Magnitude별 Measurement
for n in [64, 256, 1024, 2048]:
    mean_ms, std_ms = bench_np_matmul(n, repeat=3)
    print(f"  n={n:5d}: {mean_ms:8.3f} ± {std_ms:6.3f} ms")


In [ ]:
# PyTorch가 있다면 CPU vs GPU 비교
try:
    import torch
    print("PyTorch CPU vs GPU Matrix곱 Comparison:\n")
    print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
    print("-" * 50)
    for n in [64, 256, 1024, 2048]:
        # CPU
        A_cpu = torch.randn(n, n)
        B_cpu = torch.randn(n, n)
        for _ in range(2): _ = A_cpu @ B_cpu  # warmup
        t0 = time.perf_counter()
        for _ in range(3):
            _ = A_cpu @ B_cpu
        cpu_ms = (time.perf_counter() - t0) / 3 * 1000

        if torch.cuda.is_available():
            A_gpu = A_cpu.cuda()
            B_gpu = B_cpu.cuda()
            for _ in range(2): _ = A_gpu @ B_gpu  # warmup
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(3):
                _ = A_gpu @ B_gpu
            torch.cuda.synchronize()
            gpu_ms = (time.perf_counter() - t0) / 3 * 1000
            speedup = cpu_ms / gpu_ms
            print(f"{n:>6} | {cpu_ms:>12.3f} | {gpu_ms:>12.3f} | {speedup:>9.2f}x")
        else:
            print(f"{n:>6} | {cpu_ms:>12.3f} | {'N/A':>12} | {'N/A':>10}")
    if not torch.cuda.is_available():
        print("\n⚠ GPU가 없습니다. Colab 런타임을 T4 GPU로 전환하세요: Runtime → Change runtime type")
except ImportError:
    print("PyTorch가 없어 이 셀은 건너뜁니다.")


## 0.5 벤치마크 방법론

이 책의 모든 벤치마크는 다음 원칙을 따른다:

1. **Warm-up**: 처음 2~3회 실행은 캐시/컴파일 효과로 느리므로 측정에서 제외
2. **반복 측정**: 5~10회 반복 후 평균·표준편차 보고
3. **동기화**: GPU 측정 시 `torch.cuda.synchronize()`로 비동기 연산 완료 대기
4. **메모리 추적**: GPU의 경우 `torch.cuda.max_memory_allocated()`로 최대 메모리 측정
5. **동일한 데이터**: CPU/GPU 모두 같은 초기화 시드 사용

이를 자동화한 헬퍼가 `llm_math.bench`에 있다.


In [ ]:
# llm_math.bench 헬퍼 사용 예시
import torch
from llm_math.bench import time_fn, format_results_table, get_device

device = get_device()
print(f"현재 디바이스: {device}")

def matmul_op(A, B):
    return A @ B

results = {}
for dev_name in ['cpu'] + (['cuda'] if torch.cuda.is_available() else []):
    n = 1024
    A = torch.randn(n, n, device=dev_name)
    B = torch.randn(n, n, device=dev_name)
    res = time_fn(matmul_op, A, B, device=dev_name, warmup=2, repeat=5)
    results[dev_name.upper()] = res

print(format_results_table(results, title="Matrix곱 (n=1024) 벤치마크"))


## 0.6 연습문제와 해설 노트북

각 장 끝에는 연습문제가 있다. 해설은 `solutions/chXX_solutions.ipynb`에 있다.

## 0.7 다음 장에서

**Ch 01. 벡터와 공간**에서는 LLM의 모든 것이 시작되는 벡터를 다룬다.
임베딩, 어텐션, 그래디언트 — 모든 것이 벡터와 공간의 언어로 표현된다.

---

### 요약
- LLM 내부를 이해하려면 수학(선형대수·미분·확률)이 필수
- 이 책은 상향식으로: 수학 → 신경망 → 어텐션 → 트랜스포머 → LLM
- 모든 노트북은 Colab에서 CPU/GPU 전환 가능
- 공통 유틸리티 `llm_math` 패키지 사용
